Connected to Python

In [ ]:
%pip install openpyxl lxml numpy
import yfinance as yf
import pandas as pd
import numpy as np
import time

print("🌐 S&P 500 및 나스닥 100 종목 리스트 수집 중...")
# 1. S&P 500 티커 수집
sp_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
sp_table = pd.read_html(sp_url)[0]
sp_tickers = sp_table['Symbol'].tolist()

# 2. 나스닥 100 티커 수집
ndx_url = 'https://en.wikipedia.org/wiki/Nasdaq-100'
ndx_table = pd.read_html(ndx_url)[4] # 나스닥 100 종목 표 위치
ndx_tickers = ndx_table['Ticker'].tolist()

# 3. 두 리스트 합치고 중복 제거, 특수문자 변환 (BRK.B -> BRK-B)
raw_tickers = list(set(sp_tickers + ndx_tickers))
tickers = [t.replace('.', '-') for t in raw_tickers]
print(f"✅ 총 {len(tickers)}개 유니크 종목 장전 완료!\n")

data_list = []
print("🏦 완전체 멀티 팩터 봇 스캔 시작! (약 10~15분 소요... 커피 한 잔 드시고 오세요!)\n")

for i, ticker_str in enumerate(tickers, 1):
    try:
        if i % 50 == 0:
            print(f"진행 상황: {i}/{len(tickers)} 스캔 중...")
            
        ticker = yf.Ticker(ticker_str)
        info = ticker.info
        
        # --- 🛡️ 방어벽 1: 거래대금 (Liquidity) ---
        avg_volume = info.get('averageVolume', 0)
        current_price = info.get('currentPrice', 0)
        dollar_volume = avg_volume * current_price
        
        # 하루 거래대금이 1,000만 달러(약 130억 원) 미만이면 탈락!
        if dollar_volume < 10000000:
            continue
            
        # 팩터 1 & 2: 가치(PER)와 우량(ROE)
        per = info.get('forwardPE', None)
        roe = info.get('returnOnEquity', None)
        
        if per is None or roe is None or per <= 0 or roe <= 0:
            continue # 적자 기업 탈락
            
        # --- 🛡️ 방어벽 2 & 팩터 3: 샤프 지수 (위험 조정 모멘텀) ---
        hist = ticker.history(period="6mo")
        if hist.empty or len(hist) < 100:
            continue
            
        # 매일의 수익률 계산 후 변동성(위험) 추출
        daily_returns = hist['Close'].pct_change().dropna()
        volatility = daily_returns.std() * np.sqrt(252) # 연간 변동성
        
        # 6개월 단순 수익률
        start_price = hist['Close'].iloc[0]
        end_price = hist['Close'].iloc[-1]
        return_6m = (end_price - start_price) / start_price
        
        # 수익률을 변동성으로 나눈 샤프 지수 대용치 (높을수록 좋음)
        sharpe_momentum = return_6m / volatility if volatility > 0 else 0
        
        data_list.append({
            'Ticker': ticker_str,
            'PER(Value)': per,
            'ROE(Quality)': roe,
            'Sharpe(Momentum)': sharpe_momentum
        })
        
        time.sleep(0.1) # 서버 차단 방지 매너타임
        
    except Exception as e:
        pass

# 데이터프레임 변환
df = pd.DataFrame(data_list)

print("\n⚙️ 스캔 완료! 미국 시장 맞춤형 비율로 점수를 계산합니다...")

# --- 🎯 퀀트 매직: 순위(Rank) 매기기 ---
df['Rank_Value'] = df['PER(Value)'].rank(ascending=True)       # 낮을수록 1등
df['Rank_Quality'] = df['ROE(Quality)'].rank(ascending=False)  # 높을수록 1등
df['Rank_Sharpe'] = df['Sharpe(Momentum)'].rank(ascending=False) # 높을수록 1등

# 퀀트 매니저들의 보편적 대형주 세팅: 가치 1 + 우량 1 + 모멘텀 1 (동일 가중)
# (나중에 본인 성향에 맞춰 이 더하기 비율을 수정할 수 있습니다)
df['Total_Score'] = df['Rank_Value'] + df['Rank_Quality'] + df['Rank_Sharpe']

# 1등부터 줄 세우기
df = df.sort_values(by='Total_Score', ascending=True).reset_index(drop=True)

# 보기 편하게 데이터 정리
df['ROE(Quality)'] = (df['ROE(Quality)'] * 100).round(2)
df['Sharpe(Momentum)'] = df['Sharpe(Momentum)'].round(2)
df['PER(Value)'] = df['PER(Value)'].round(2)

# 최종 엑셀용 데이터
final_columns = ['Ticker', 'PER(Value)', 'ROE(Quality)', 'Sharpe(Momentum)', 'Total_Score']
final_df = df[final_columns]

# 엑셀 저장
excel_filename = "US_Super_Quant_List.xlsx"
final_df.to_excel(excel_filename, index=False)

print(f"\n✅ 완성! 최고 존엄 1위 종목은 [{final_df.iloc[0]['Ticker']}] 입니다!")
print(f"좌측 파일 목록에서 [{excel_filename}]을 확인하세요!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


🌐 S&P 500 및 나스닥 100 종목 리스트 수집 중...


ImportError: `Import lxml` failed.  Use pip or conda to install the lxml package.

_Connecting to .venv (3.12.4.final.0) (Python 3.12.4)..._

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import time

print("🌐 S&P 500 및 나스닥 100 종목 리스트 수집 중...")
# 1. S&P 500 티커 수집
sp_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
sp_table = pd.read_html(sp_url)[0]
sp_tickers = sp_table['Symbol'].tolist()

# 2. 나스닥 100 티커 수집
ndx_url = 'https://en.wikipedia.org/wiki/Nasdaq-100'
ndx_table = pd.read_html(ndx_url)[4] # 나스닥 100 종목 표 위치
ndx_tickers = ndx_table['Ticker'].tolist()

# 3. 두 리스트 합치고 중복 제거, 특수문자 변환 (BRK.B -> BRK-B)
raw_tickers = list(set(sp_tickers + ndx_tickers))
tickers = [t.replace('.', '-') for t in raw_tickers]
print(f"✅ 총 {len(tickers)}개 유니크 종목 장전 완료!\n")

data_list = []
print("🏦 완전체 멀티 팩터 봇 스캔 시작! (약 10~15분 소요... 커피 한 잔 드시고 오세요!)\n")

for i, ticker_str in enumerate(tickers, 1):
    try:
        if i % 50 == 0:
            print(f"진행 상황: {i}/{len(tickers)} 스캔 중...")
            
        ticker = yf.Ticker(ticker_str)
        info = ticker.info
        
        # --- 🛡️ 방어벽 1: 거래대금 (Liquidity) ---
        avg_volume = info.get('averageVolume', 0)
        current_price = info.get('currentPrice', 0)
        dollar_volume = avg_volume * current_price
        
        # 하루 거래대금이 1,000만 달러(약 130억 원) 미만이면 탈락!
        if dollar_volume < 10000000:
            continue
            
        # 팩터 1 & 2: 가치(PER)와 우량(ROE)
        per = info.get('forwardPE', None)
        roe = info.get('returnOnEquity', None)
        
        if per is None or roe is None or per <= 0 or roe <= 0:
            continue # 적자 기업 탈락
            
        # --- 🛡️ 방어벽 2 & 팩터 3: 샤프 지수 (위험 조정 모멘텀) ---
        hist = ticker.history(period="6mo")
        if hist.empty or len(hist) < 100:
            continue
            
        # 매일의 수익률 계산 후 변동성(위험) 추출
        daily_returns = hist['Close'].pct_change().dropna()
        volatility = daily_returns.std() * np.sqrt(252) # 연간 변동성
        
        # 6개월 단순 수익률
        start_price = hist['Close'].iloc[0]
        end_price = hist['Close'].iloc[-1]
        return_6m = (end_price - start_price) / start_price
        
        # 수익률을 변동성으로 나눈 샤프 지수 대용치 (높을수록 좋음)
        sharpe_momentum = return_6m / volatility if volatility > 0 else 0
        
        data_list.append({
            'Ticker': ticker_str,
            'PER(Value)': per,
            'ROE(Quality)': roe,
            'Sharpe(Momentum)': sharpe_momentum
        })
        
        time.sleep(0.1) # 서버 차단 방지 매너타임
        
    except Exception as e:
        pass

# 데이터프레임 변환
df = pd.DataFrame(data_list)

print("\n⚙️ 스캔 완료! 미국 시장 맞춤형 비율로 점수를 계산합니다...")

# --- 🎯 퀀트 매직: 순위(Rank) 매기기 ---
df['Rank_Value'] = df['PER(Value)'].rank(ascending=True)       # 낮을수록 1등
df['Rank_Quality'] = df['ROE(Quality)'].rank(ascending=False)  # 높을수록 1등
df['Rank_Sharpe'] = df['Sharpe(Momentum)'].rank(ascending=False) # 높을수록 1등

# 퀀트 매니저들의 보편적 대형주 세팅: 가치 1 + 우량 1 + 모멘텀 1 (동일 가중)
# (나중에 본인 성향에 맞춰 이 더하기 비율을 수정할 수 있습니다)
df['Total_Score'] = df['Rank_Value'] + df['Rank_Quality'] + df['Rank_Sharpe']

# 1등부터 줄 세우기
df = df.sort_values(by='Total_Score', ascending=True).reset_index(drop=True)

# 보기 편하게 데이터 정리
df['ROE(Quality)'] = (df['ROE(Quality)'] * 100).round(2)
df['Sharpe(Momentum)'] = df['Sharpe(Momentum)'].round(2)
df['PER(Value)'] = df['PER(Value)'].round(2)

# 최종 엑셀용 데이터
final_columns = ['Ticker', 'PER(Value)', 'ROE(Quality)', 'Sharpe(Momentum)', 'Total_Score']
final_df = df[final_columns]

# 엑셀 저장
excel_filename = "US_Super_Quant_List.xlsx"
final_df.to_excel(excel_filename, index=False)

print(f"\n✅ 완성! 최고 존엄 1위 종목은 [{final_df.iloc[0]['Ticker']}] 입니다!")
print(f"좌측 파일 목록에서 [{excel_filename}]을 확인하세요!")

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import time

print("🌐 S&P 500 및 나스닥 100 종목 리스트 수집 중...")
# 1. S&P 500 티커 수집
sp_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
sp_table = pd.read_html(sp_url)[0]
sp_tickers = sp_table['Symbol'].tolist()

# 2. 나스닥 100 티커 수집
ndx_url = 'https://en.wikipedia.org/wiki/Nasdaq-100'
ndx_table = pd.read_html(ndx_url)[4] # 나스닥 100 종목 표 위치
ndx_tickers = ndx_table['Ticker'].tolist()

# 3. 두 리스트 합치고 중복 제거, 특수문자 변환 (BRK.B -> BRK-B)
raw_tickers = list(set(sp_tickers + ndx_tickers))
tickers = [t.replace('.', '-') for t in raw_tickers]
print(f"✅ 총 {len(tickers)}개 유니크 종목 장전 완료!\n")

data_list = []
print("🏦 완전체 멀티 팩터 봇 스캔 시작! (약 10~15분 소요... 커피 한 잔 드시고 오세요!)\n")

for i, ticker_str in enumerate(tickers, 1):
    try:
        if i % 50 == 0:
            print(f"진행 상황: {i}/{len(tickers)} 스캔 중...")
            
        ticker = yf.Ticker(ticker_str)
        info = ticker.info
        
        # --- 🛡️ 방어벽 1: 거래대금 (Liquidity) ---
        avg_volume = info.get('averageVolume', 0)
        current_price = info.get('currentPrice', 0)
        dollar_volume = avg_volume * current_price
        
        # 하루 거래대금이 1,000만 달러(약 130억 원) 미만이면 탈락!
        if dollar_volume < 10000000:
            continue
            
        # 팩터 1 & 2: 가치(PER)와 우량(ROE)
        per = info.get('forwardPE', None)
        roe = info.get('returnOnEquity', None)
        
        if per is None or roe is None or per <= 0 or roe <= 0:
            continue # 적자 기업 탈락
            
        # --- 🛡️ 방어벽 2 & 팩터 3: 샤프 지수 (위험 조정 모멘텀) ---
        hist = ticker.history(period="6mo")
        if hist.empty or len(hist) < 100:
            continue
            
        # 매일의 수익률 계산 후 변동성(위험) 추출
        daily_returns = hist['Close'].pct_change().dropna()
        volatility = daily_returns.std() * np.sqrt(252) # 연간 변동성
        
        # 6개월 단순 수익률
        start_price = hist['Close'].iloc[0]
        end_price = hist['Close'].iloc[-1]
        return_6m = (end_price - start_price) / start_price
        
        # 수익률을 변동성으로 나눈 샤프 지수 대용치 (높을수록 좋음)
        sharpe_momentum = return_6m / volatility if volatility > 0 else 0
        
        data_list.append({
            'Ticker': ticker_str,
            'PER(Value)': per,
            'ROE(Quality)': roe,
            'Sharpe(Momentum)': sharpe_momentum
        })
        
        time.sleep(0.1) # 서버 차단 방지 매너타임
        
    except Exception as e:
        pass

# 데이터프레임 변환
df = pd.DataFrame(data_list)

print("\n⚙️ 스캔 완료! 미국 시장 맞춤형 비율로 점수를 계산합니다...")

# --- 🎯 퀀트 매직: 순위(Rank) 매기기 ---
df['Rank_Value'] = df['PER(Value)'].rank(ascending=True)       # 낮을수록 1등
df['Rank_Quality'] = df['ROE(Quality)'].rank(ascending=False)  # 높을수록 1등
df['Rank_Sharpe'] = df['Sharpe(Momentum)'].rank(ascending=False) # 높을수록 1등

# 퀀트 매니저들의 보편적 대형주 세팅: 가치 1 + 우량 1 + 모멘텀 1 (동일 가중)
# (나중에 본인 성향에 맞춰 이 더하기 비율을 수정할 수 있습니다)
df['Total_Score'] = df['Rank_Value'] + df['Rank_Quality'] + df['Rank_Sharpe']

# 1등부터 줄 세우기
df = df.sort_values(by='Total_Score', ascending=True).reset_index(drop=True)

# 보기 편하게 데이터 정리
df['ROE(Quality)'] = (df['ROE(Quality)'] * 100).round(2)
df['Sharpe(Momentum)'] = df['Sharpe(Momentum)'].round(2)
df['PER(Value)'] = df['PER(Value)'].round(2)

# 최종 엑셀용 데이터
final_columns = ['Ticker', 'PER(Value)', 'ROE(Quality)', 'Sharpe(Momentum)', 'Total_Score']
final_df = df[final_columns]

# 엑셀 저장
excel_filename = "US_Super_Quant_List.xlsx"
final_df.to_excel(excel_filename, index=False)

print(f"\n✅ 완성! 최고 존엄 1위 종목은 [{final_df.iloc[0]['Ticker']}] 입니다!")
print(f"좌측 파일 목록에서 [{excel_filename}]을 확인하세요!")